# E2-T04 — model and data pilot

Sections 1–3 can run before the data gate; CPU is sufficient for selection planning. Sections 4–6 require an **L4 GPU** and a valid `manifests/data_gate_approval.json` in Drive. Run cells in order. M1 and M2 execute in separate processes so only one model occupies the GPU at a time. Re-running a completed model cell verifies and reuses its receipt.


## 1. Mount Drive and locate the private project folder


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
SHARED = Path('/content/drive/MyDrive/CMPE253-Project').resolve()
assert SHARED.is_dir(), f'Missing shared folder: {SHARED}'
DATA_GATE_READY = (SHARED / 'manifests/data_gate_approval.json').is_file()
print('Data-gate receipt present:', DATA_GATE_READY)
print(SHARED)


## 2. Pull the pushed code and install the locked GPU environment


In [ ]:
import subprocess, sys

REPO = Path('/content/cmpe253-llm-demographic-bias-audit')
URL = 'https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', URL, str(REPO)], check=True)
assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['make', 'setup-colab'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], cwd=REPO, text=True))


## 3. Freeze or verify the pilot selection

Expected: 50 CAB families and 100 BBQ items. An existing identical selection is reused; a changed selection fails visibly.


In [ ]:
subprocess.run(['make', 'pilot', f'ROOT={SHARED}', 'ACTION=plan'], cwd=REPO, check=True)
import json
selection = json.loads((SHARED / 'manifests/pilot/selection.json').read_text())
assert selection['cab']['families'] == 50
assert selection['bbq']['items'] == 100 and selection['bbq']['permutations'] == 300
print({'cab_families': selection['cab']['families'], 'cab_prompts': selection['cab']['prompts'], 'bbq_items': selection['bbq']['items']})


## 4. Run M1

This runs CAB baseline, BBQ baseline, CAB mitigation, and BBQ mitigation. It may take substantial time. If Colab disconnects, reconnect and rerun this cell; completed checkpoints are reused.


In [ ]:
assert DATA_GATE_READY, 'Stop here until the data-gate receipt is created'
subprocess.run(['make', 'pilot', f'ROOT={SHARED}', 'ACTION=run-model', 'MODEL=M1'], cwd=REPO, check=True)


## 5. Run M2


In [ ]:
assert DATA_GATE_READY, 'Stop here until Abhinand creates the data-gate receipt'
subprocess.run(['make', 'pilot', f'ROOT={SHARED}', 'ACTION=run-model', 'MODEL=M2'], cwd=REPO, check=True)


## 6. Reconcile and inspect the report

Expected status: `ready_for_operator_review`, eight completed run cells, zero errors, and one GPU class within each model. This does not approve the full-run gate. Vineet must later inspect the private 20-row output packet and record the decision.


In [ ]:
subprocess.run(['make', 'pilot', f'ROOT={SHARED}', 'ACTION=report'], cwd=REPO, check=True)
report = json.loads((SHARED / 'manifests/pilot/pilot_report.json').read_text())
assert report['status'] == 'ready_for_operator_review'
assert report['run_count'] == 8 and all(report['checks'].values())
print(json.dumps({'status': report['status'], 'checks': report['checks'], 'projected_full_seconds_by_model': report['projected_full_seconds_by_model'], 'review_packet': report['review_packet_path']}, indent=2))
